# DataCrunch 2

## Challenge Overview

Datacrunch uses the quantitative research of the CrunchDAO to manage its systematic market-neutral portfolio. DataCrunch built a dataset covering thousands of publicly traded U.S companies.

The long-term strategic goal of the fund is capital appreciation by capturing idiosyncratic return at low volatility.

In order to achieve this goal, Datacrunch needs the community to assess the relative performance of all assets in a subset of the [Russell 3000](https://www.investopedia.com/terms/r/russell_3000.asp) universe. In other words, DataCrunch is expecting your model to maximise the correlation to the constituent of its investment universe.

# Setup

The first steps to get started are:
1. Get the setup command
2. Execute it in the cell below

### >> https://hub.crunchdao.com/competitions/datacrunch-2/submit/notebook

![Reveal token](https://raw.githubusercontent.com/crunchdao/competitions/refs/heads/master/documentation/animations/reveal-token.gif)

In [ ]:
# Install the Crunch CLI
%pip install --upgrade crunch-cli

# Setup your local environment
!crunch setup-notebook datacrunch-2 --token aaaabbbbccccddddeeeeffff

# Your model

## Setup

In [ ]:
# Imports
import os

import joblib  # == 1.3.2
import lightgbm as lgb
import numpy as np
import pandas as pd  # == 2.1.0
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import root_mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


In [ ]:
import crunch

# Load the Crunch Toolings
crunch_tools = crunch.load_notebook()

## Strategy Implementation

### Utilities

Function used in both `train()` and `infer()`.

In [ ]:
def get_model_path(
    model_directory_path: str,
):
    return os.path.join(
        model_directory_path,
        f"model.joblib"
    )

get_model_path("resources")

In [ ]:
def get_model_path(
    model_directory_path: str,
    backend: str | None = None,
):
    filename = "model.joblib" if backend is None else f"model_{backend}.joblib"
    return os.path.join(
        model_directory_path,
        filename,
    )


def save_model_artifacts(
    model,
    model_directory_path: str,
    backend: str,
):
    named_model_path = get_model_path(model_directory_path, backend)
    alias_model_path = get_model_path(model_directory_path)
    joblib.dump(model, named_model_path)
    joblib.dump(model, alias_model_path)
    return named_model_path


def get_feature_columns(
    X: pd.DataFrame,
):
    return [
        column
        for column in X.columns
        if column.startswith("Feature_")
    ]


def split_moons(
    X: pd.DataFrame,
    validation_size: float = 0.1,
    gap: int = 4,
):
    moons = sorted(X["moon"].unique())
    if len(moons) <= gap + 2:
        raise ValueError("Not enough moons to create a validation split with the requested gap.")

    validation_count = max(1, int(len(moons) * validation_size))
    validation_start_index = max(gap + 1, len(moons) - validation_count)
    validation_start_moon = int(moons[validation_start_index])
    train_end_moon = int(moons[validation_start_index - gap - 1])

    if train_end_moon >= validation_start_moon:
        raise ValueError("Invalid moon split: training period overlaps validation period.")

    return train_end_moon, validation_start_moon


def load_data():
    X_train = pd.read_parquet("data/X.reduced.parquet")
    y_train = pd.read_parquet("data/y.reduced.parquet")
    return X_train, y_train


def build_ridge_pipeline(alpha: float):
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("variance_threshold", VarianceThreshold(threshold=0.0)),
            ("scaler", StandardScaler()),
            ("ridge", Ridge(alpha=alpha)),
        ]
    )


def predict_with_model(model, X_features: pd.DataFrame):
    if isinstance(model, lgb.Booster):
        return model.predict(X_features.astype(np.float32, copy=False))

    return model.predict(X_features)


### The `train()` Function

In this function, you build and train your model for making inferences on the test data. Your model must be stored in the `model_directory_path`.

This function will be called in a frequency that is defined by your `train frequency` parameter that you will define when deploying your model on the Crunch platform.

In [ ]:
def train(
    X_train: pd.DataFrame,
    y_train: pd.DataFrame,
    model_directory_path: str,
    backend: str = "lightgbm",
    use_cuda: bool = False,
) -> None:
    feature_columns = get_feature_columns(X_train)

    train_end_moon, validation_start_moon = split_moons(X_train)
    moon_values = X_train["moon"].to_numpy()
    train_mask = moon_values <= train_end_moon
    validation_mask = moon_values >= validation_start_moon

    if backend == "ridge":
        x_train_frame = X_train.loc[train_mask, feature_columns]
        y_train_frame = y_train.loc[train_mask, "target"]
        x_validation_frame = X_train.loc[validation_mask, feature_columns]
        y_validation_frame = y_train.loc[validation_mask, "target"]

        model = build_ridge_pipeline(alpha=1.0)
        model.fit(x_train_frame, y_train_frame)

        validation_prediction = pd.Series(model.predict(x_validation_frame)).clip(-1, 1)
        validation_score = validation_prediction.corr(pd.Series(y_validation_frame).reset_index(drop=True))
        validation_rmse = root_mean_squared_error(y_validation_frame, validation_prediction)
        model_path = save_model_artifacts(model, model_directory_path, "ridge")

    elif backend == "lightgbm":
        x_train_frame = X_train.loc[train_mask, feature_columns].astype(np.float32, copy=False)
        y_train_frame = y_train.loc[train_mask, "target"].astype(np.float32, copy=False)
        x_validation_frame = X_train.loc[validation_mask, feature_columns].astype(np.float32, copy=False)
        y_validation_frame = y_train.loc[validation_mask, "target"].astype(np.float32, copy=False)

        train_dataset = lgb.Dataset(x_train_frame, label=y_train_frame, free_raw_data=False)
        validation_dataset = lgb.Dataset(
            x_validation_frame,
            label=y_validation_frame,
            reference=train_dataset,
            free_raw_data=False,
        )

        params = {
            "objective": "regression",
            "metric": "rmse",
            "learning_rate": 0.05,
            "num_leaves": 63,
            "max_depth": -1,
            "feature_fraction": 0.8,
            "bagging_fraction": 0.8,
            "bagging_freq": 1,
            "min_child_samples": 20,
            "lambda_l1": 0.0,
            "lambda_l2": 0.0,
            "seed": 42,
            "verbosity": -1,
        }

        def fit_with_device(device_type: str):
            current_params = dict(params)
            current_params["device_type"] = device_type
            return lgb.train(
                params=current_params,
                train_set=train_dataset,
                num_boost_round=300,
                valid_sets=[train_dataset, validation_dataset],
                valid_names=["train", "valid"],
                callbacks=[lgb.log_evaluation(period=30)],
            )

        try:
            model = fit_with_device("cuda" if use_cuda else "cpu")
        except Exception as exc:
            if use_cuda and ("CUDA Tree Learner was not enabled" in str(exc) or "No OpenCL device" in str(exc)):
                model = fit_with_device("cpu")
            else:
                raise

        best_iteration = model.best_iteration if model.best_iteration and model.best_iteration > 0 else 300
        validation_prediction = pd.Series(model.predict(x_validation_frame, num_iteration=best_iteration)).clip(-1, 1)
        validation_score = validation_prediction.corr(pd.Series(y_validation_frame).reset_index(drop=True))
        validation_rmse = root_mean_squared_error(y_validation_frame, validation_prediction)
        model_path = save_model_artifacts(model, model_directory_path, "lightgbm")

    else:
        raise ValueError("backend must be either 'ridge' or 'lightgbm'.")

    print(
        {
            "backend": backend,
            "train_rows": int(train_mask.sum()),
            "validation_rows": int(validation_mask.sum()),
            "validation_score": None if pd.isna(validation_score) else float(validation_score),
            "validation_rmse": float(validation_rmse),
            "model_path": model_path,
        }
    )

### The `infer()` Function

In the inference function, your trained model (if any) is loaded and used to make predictions on test data.

This function will be called on every `moon` of the `Out-Of-Sample`.

In [ ]:
def infer(
    X_test: pd.DataFrame,
    model_directory_path: str,
    backend: str = "lightgbm",
) -> pd.DataFrame:
    prediction = X_test[["id", "moon"]].copy()

    model_path = get_model_path(model_directory_path, backend)
    model = joblib.load(model_path)

    feature_columns = get_feature_columns(X_test)
    prediction["prediction"] = predict_with_model(model, X_test[feature_columns])
    prediction["prediction"] = prediction["prediction"].clip(-1, 1)

    return prediction

## Local testing

To make sure your `train()` and `infer()` function are working properly, you can call the `crunch.test()` function that will reproduce the cloud environment locally. <br />
Even if it is not perfect, it should give you a quick idea if your model is working properly.

In [ ]:
crunch_tools.test(
    # Uncomment to disable the forced first train
    # force_first_train=False,
    force_first_train=True,

    # Uncomment to set the training frequency
    # train_frequency=2,  # train every 2 moons
    train_frequency=0,

    # Uncomment to disable the determinism check
    # no_determinism_check=True,
)

## Results

Once the local tester is done, you can preview the result stored in `prediction/prediction.parquet`.

### Local scoring

You can call the function that the system uses to estimate your score locally.

A [Pearson correlation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corr.html) will be computed against the **targets**.

**Note**:
- If all predictions are constant, the correlation will be undefined. In this case, the score will be set to `0`.
- Predictions must be between `-1` and `1`.